In [0]:
# Path 1 not so exciting
# import pandas as pd
# df = pd.read_csv('/Workspace/Users/josurekamotho@softserve.academy/Lab1/house_prices.csv')
# df.head()

In [0]:
%sql
-- Checking if the schema exists
-- SHOW SCHEMAS IN dbr_dev;
SHOW TABLES IN dbr_dev.joshuandegwa7447;

In [0]:
%sql
-- Creating my schema
CREATE SCHEMA IF NOT EXISTS dbr_dev.joshuandegwa7447;

Manually created Volume in students in the schema, and uploaded the students performace file. 

In [0]:
# 1. Read the CSV file
file_path = "/Volumes/dbr_dev/joshuandegwa7447/students/StudentsPerformance.csv"
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(file_path)

# 2. Clean the column names automatically by replacing spaces with underscores
for col_name in df.columns:
    cleaned_name = col_name.replace(" ", "_")
    df = df.withColumnRenamed(col_name, cleaned_name)

# 3. Save to your Delta table (this will now succeed!)
target_table = "dbr_dev.joshuandegwa7447.StudentsPerformance"
df.write.format("delta").mode("overwrite").saveAsTable(target_table)

print("Success! Table created with clean column names.")

In [0]:
%sql
-- Checking if the table was successfully created
SHOW TABLES IN dbr_dev.joshuandegwa7447;
SELECT * FROM dbr_dev.joshuandegwa7447.StudentsPerformance LIMIT 10;

In [0]:
# Load your official students table
df = spark.read.table("dbr_dev.joshuandegwa7447.StudentsPerformance")

# Take a quick peek to confirm columns (like math_score, reading_score, etc.)
display(df)

In [0]:
# Selecting relevant columns for performance analysis
scores_df = df.select("gender","lunch", "parental_level_of_education", "test_preparation_course", "math_score", "reading_score", "writing_score")
display(scores_df)

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql import functions as F

# Group by parental level of education and count students
edu_counts_df = df.groupBy("parental_level_of_education").count().orderBy("count", ascending=False)

# Display the data frame
display(edu_counts_df)

Databricks visualization. Run in Databricks to view.

In [0]:
# Group by parental level of education and sum of math score
from pyspark.sql import functions as F
edu_counts_df = df.groupBy("parental_level_of_education","gender").agg(F.sum('math_score').alias("Total_Math")).orderBy("parental_level_of_education", "Total_Math", ascending=False)

# Display the data frame
display(edu_counts_df)

Databricks visualization. Run in Databricks to view.

In [0]:
# Define a list of categories to match
higher_ed_degrees = ["bachelor's degree", "master's degree"]

# Filter using the list
higher_ed_df = df.filter(df["parental_level_of_education"].isin(higher_ed_degrees))

display(higher_ed_df)


## Chaining multiple filters together
high_performing_no_prep = df.filter(
    (df["gender"] == "female") & 
    (df["test_preparation_course"] == "none") & 
    (df["math_score"] > 80)
)

display(high_performing_no_prep)

Databricks visualization. Run in Databricks to view.

To do joins I decided to create dummy df

In [0]:
# Define a list of categories to match
higher_ed_degrees = ["bachelor's degree", "master's degree"]

# Filter using the list
higher_ed_df = df.filter(df["parental_level_of_education"].isin(higher_ed_degrees))

display(higher_ed_df)


## Chaining multiple filters together
high_performing_no_prep = df.filter(
    (df["gender"] == "male") & 
    (df["test_preparation_course"] == "none") & 
    (df["math_score"] > 80)
)

display(high_performing_no_prep)

Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql.functions import broadcast

# Create the Parent Demographics Lookup Table
parent_demographics_data = [
    ("master's degree", "Kraków", "Małopolskie", 11500),
    ("bachelor's degree", "Warsaw", "Mazowieckie", 10200),
    ("associate's degree", "Wrocław", "Dolnośląskie", 8800),
    ("some college", "Gdańsk", "Pomorskie", 7900),
    ("high school", "Poznań", "Wielkopolskie", 6400),
    ("some high school", "Łódź", "Łódzkie", 5200)
]

parent_cols = ["parental_level_of_education", "city", "voivodeship", "avg_salary_pln"]
parent_lookup_df = spark.createDataFrame(parent_demographics_data, parent_cols)

In [0]:
# Perform the join
enriched_df = df.join(
    broadcast(parent_lookup_df), 
    on="parental_level_of_education", 
    how="inner"
)

# View the result with your new salary dimension attached
display(enriched_df.select("parental_level_of_education", "city", "avg_salary_pln", "math_score", "reading_score"))

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
salary_vs_scores = enriched_df.groupBy( "city", "avg_salary_pln").agg(
    F.round(F.avg("math_score"), 1).alias("Avg_Math_Score"),
    F.round(F.avg("reading_score"), 1).alias("Avg_Reading_Score")
).orderBy("avg_salary_pln", ascending=True)

display(salary_vs_scores)

In [0]:
# Combined regional performance and economic report
regional_report_df = enriched_df.groupBy("voivodeship", "city").agg(
    F.round(F.avg("math_score"), 1).alias("Avg_Math_Score"),
    F.round(F.avg("avg_salary_pln"), 0).alias("Avg_Salary_PLN"),
    F.count("math_score").alias("Student_Count")
).orderBy("Avg_Math_Score", ascending=False)

display(regional_report_df)

Load data from an external API. 

In [0]:
%pip install yfinance

In [0]:
from datetime import datetime
import yfinance as yf
import pandas as pd
from pyspark.sql.types import *

def download_and_save_ticker_data_spark():
    # Define your tickers (Global tech giants & some emerging market plays)
    tickers = ["AAPL", "AMZN", "TSLA", "JMIA", "SSL", "AU"]
    
    # Dynamic dates (Jan 1st of current year to today)
    start_date = f"{datetime.now().year}-01-01"
    end_date = datetime.now().strftime("%Y-%m-%d")
    
    print(f"Downloading data from {start_date} to {end_date} via yfinance...")
    
    # 1. Fetch data into a multi-index Pandas dataframe on the driver node
    raw_data = yf.download(tickers, start=start_date, end=end_date, group_by='ticker')
    
    combined_pdf = pd.DataFrame()
    
    # 2. Reshape into flat tabular structure using your loop logic
    for ticker in tickers:
        if ticker in raw_data.columns.levels[0]: # Defensive check if data exists
            ticker_df = raw_data[ticker].dropna(how='all').copy()
            ticker_df['Ticker'] = ticker
            ticker_df = ticker_df.reset_index()
            
            # Ensure column names are clean strings (yfinance sometimes outputs tuple columns)
            ticker_df.columns = [str(col) if isinstance(col, tuple) else col for col in ticker_df.columns]
            
            combined_pdf = pd.concat([combined_pdf, ticker_df], ignore_index=True)
            
    # Clean up column names for Spark compatibility (remove spaces/special chars)
    combined_pdf.columns = [col.replace(" ", "_").strip() for col in combined_pdf.columns]
    
    # 3. Convert from Pandas DataFrame to a Distributed Spark DataFrame
    # Casting 'Date' to string in Pandas handles timezone compatibility cleanly during conversion
    if 'Date' in combined_pdf.columns:
        combined_pdf['Date'] = combined_pdf['Date'].dt.strftime('%Y-%m-%d')
        
    spark_market_df = spark.createDataFrame(combined_pdf)
    display(spark_market_df)
    # 4. Write it directly to your delta lake warehouse!
    # Overwrite means it refreshes the table with current year-to-date data every time you run it
    target_table = "dbr_dev.joshuandegwa7447.global_market_data"
    spark_market_df.write.format("delta").mode("overwrite").saveAsTable(target_table)
    
    print(f"\nSuccess! Data written to Delta table: {target_table}")
    print(f"Total rows managed by Spark: {spark_market_df.count()}")

# Execute the function inside your notebook cell
download_and_save_ticker_data_spark()

In [0]:
%sql
SELECT * FROM dbr_dev.joshuandegwa7447.global_market_data LIMIT 10;

In [0]:
# Load your official markets table
df = spark.read.table("dbr_dev.joshuandegwa7447.global_market_data")
display(df)

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.